# N2/N6: chạy RTG (bản B, C, D) cho CUB, Stanford Cars, FGVC-Aircraft

Bật GPU (Settings → Accelerator → GPU T4 x2) và Internet (Session options). RTG dùng CLIP ViT-H-14 (laion2B), tải từ HuggingFace.
Kết quả: `rtg/{ds}_{ver}.pkl` + `rtg/{ds}_{ver}.json` (cấu hình, hash lexicon, thời gian chạy). Cache embedding lexicon để riêng ở `/kaggle/working/rtg_cache` (không nén vào zip).

- **N2 (bản B):** chạy ô 1 → 2 → 3 → 4 → 7.
- **N6 (bản C, D):** chạy ô 1 → 2 → 5 → 6 → 7.

In [ ]:
REPO_URL = "https://github.com/tanq1965/gcd-text-leak.git"
BRANCH = "feat/n2-rtg"  # đổi thành main sau khi merge PR #5
REPO = "/kaggle/working/gcd-text-leak"
CACHE = "/kaggle/working/rtg_cache"
!git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log -1 --format="%h %s"
!pip install -q open_clip_torch ftfy regex
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
import os
CUB_ROOT = "/kaggle/input/datasets/wenewone/cub2002011"
CARS_ROOT = "/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars"
AIRCRAFT_ROOT = "/kaggle/input/datasets/seryouxblaster764/fgvc-aircraft/fgvc-aircraft-2013b/fgvc-aircraft-2013b"
ROOTS = {"cub": CUB_ROOT, "scars": CARS_ROOT, "aircraft": AIRCRAFT_ROOT}
import pandas as pd
for ds, root in ROOTS.items():  # mọi ảnh trong split phải tồn tại
    df = pd.read_csv(f"splits/{ds}.csv", dtype=str)
    missing = [p for p in df.path if not os.path.exists(os.path.join(root, p))]
    print(ds, len(df), "ảnh, thiếu", len(missing), missing[:3])

## N2 – Smoke test: 256 ảnh CUB bản B

In [ ]:
!python -m src.rtg --dataset cub --version B --data-root {CUB_ROOT} --cache-dir {CACHE} --fp16 --limit 256

## N2 – bản B cho 3 tập

In [ ]:
for ds in ["cub", "aircraft", "scars"]:
    !python -m src.rtg --dataset {ds} --version B --data-root {ROOTS[ds]} --cache-dir {CACHE} --fp16

## N6 – Lexicon C, D
Nếu `main` chưa có lexicon của N3 (PR #4 chưa merge), lấy tạm từ nhánh `feat/n3-lexicon`. In ra số dòng để đối chiếu với audit của N3.

In [ ]:
LEXICON_BRANCH = "feat/n3-lexicon"
if not os.path.exists("lexicon/cub/C/attributes.csv"):
    !git fetch -q origin {LEXICON_BRANCH}
    !git checkout FETCH_HEAD -- lexicon names
!git log -1 --format="lexicon từ: %h %s" FETCH_HEAD 2>/dev/null || true
import csv
for ds in ROOTS:
    for ver in ["C", "D"]:
        n = {f: len(list(csv.reader(open(f"lexicon/{ds}/{ver}/{f}", encoding="utf-8-sig", newline=""))))
             for f in ["tags.csv", "attributes.csv"]}
        print(ds, ver, n)

## N6 – Smoke test bản D (không tag) rồi chạy C, D cho 3 tập

In [ ]:
!python -m src.rtg --dataset cub --version D --data-root {CUB_ROOT} --cache-dir {CACHE} --fp16 --limit 64
for ver in ["C", "D"]:
    for ds in ["cub", "aircraft", "scars"]:
        !python -m src.rtg --dataset {ds} --version {ver} --data-root {ROOTS[ds]} --cache-dir {CACHE} --fp16

## Kiểm tra và nén kết quả (chỉ `.pkl` + `.json`, không có cache)

In [ ]:
import pickle, json, glob, zipfile
files = sorted(f for f in glob.glob("rtg/*") if f.endswith((".pkl", ".json")) and "_limit" not in f)
for p in files:
    if p.endswith(".pkl"):
        r = pickle.load(open(p, "rb")); m = json.load(open(p[:-4] + ".json"))
        print(p, len(r), m["seconds"], "tags_rows:", m["tags_rows"], "no-tag:", m["images_without_tags"]); print("   ", r[0])
with zipfile.ZipFile("/kaggle/working/rtg_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p)
print("Đã nén", len(files), "file vào /kaggle/working/rtg_out.zip")